# 05 — Predict the upcoming season → `futures/futures_predictions.csv`

**Runs only on a `GO`/`GO-TIER-B` verdict AND a passing §7 gate A.** The only notebook that writes the artifact
the live page reads.

The CSV is deliberately lightweight: the page must render it with pandas + Streamlit alone, with no
model, no simulator, and no training dependency in the deployed runtime.

**Planned schema** (final column list pinned in the implementation, and validated by the page's
tests):

| column | meaning |
|---|---|
| `season` | predicted season |
| `team` | nflverse franchise abbreviation |
| `proj_wins` | projected regular-season wins (ties at 0.5) |
| `p10`, `p50`, `p90` | win-distribution quantiles from the `03` simulation |
| `win_total_line` | posted preseason line, or blank when none is available |
| `book`, `line_as_of` | provenance of that line |
| `p_over`, `p_under`, `p_push` | simulated probabilities; sum to 1.0 per row |
| `generated_at`, `model_version`, `audit_verdict` | provenance stamped into every row |

**Language fence (PREREGISTRATION §7).** No `bet`, `edge`, `lock`, `value`, `play`, or confidence
tier appears in this artifact or on the page unless gate **C** has passed and the passing evidence
is cited on the same surface. Under §10 Amendment 1 the current data is Tier B only - gate C is
unreachable from an archived consensus with no named book, so the fence is fully binding. Until then the CSV carries a projection and its distribution, and the
page says what the backtest showed — including, if that is the finding, that the model does not
beat the market.


> **STATUS: SCAFFOLD — no implementation.** The sections below are the planned structure, frozen for review before any code is written. Each will follow the repo's markdown → code → inline-test convention.

```bash
papermill futures/season_team_totals/05_predict_futures.ipynb /tmp/out.ipynb
```

## Parameters

In [ ]:
AUDIT_PATH    = None
MODEL_PATH    = None    # None -> futures/models/win_totals_model.pkl
TARGET_SEASON = None    # None -> the audit's predict season
N_SIMS        = 20000
SEED          = 20260802
WRITE_ARTIFACTS = True
RUN_TESTS     = True

## Planned sections

**Section 1 — Gate** — Audit GO + gate A; abort otherwise. Assert the target season has a published schedule and NO results (predicting a season in progress is a different task).

**Section 2 — Build the predict-season feature row** — Same builder as `01`, same pinned feature order, asserted against `model_metadata.json`.

**Section 3 — Point estimate + simulated distribution** — Model prediction and the seeded Monte Carlo from `03`.

**Section 4 — Attach the current line** — Join the current posted lines where available. A missing line blanks the market columns — it never falls back to a stale or reconstructed number.

**Section 5 — Artifact tests** — Schema exactly as pinned; 32 rows; probabilities sum to 1.0 per row within 1e-9; Σ proj_wins ≈ games scheduled ÷ 2 × 2 (league conservation); quantiles ordered p10 ≤ p50 ≤ p90; provenance columns non-null; a forbidden-language scan over every string cell and column name.

**Section 6 — Write `futures/futures_predictions.csv`** — Lightweight, page-readable, stamped.

## Not implemented

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAFFOLD — not implemented yet. Only `00_data_audit.ipynb` carries code today
# (Joseph reviews the audit before anything downstream is built), and the audit's
# current verdict is the gate this notebook would have to clear first.
#
# When implemented, this cell becomes the §5 gate: read futures/artifacts/data_audit.json,
# refuse to run on a NO-GO verdict, and read the FROZEN fold sets from it (headline + the A1.4 strict-subset sensitivity, which every reported number must carry) rather than
# choosing folds here.
# ─────────────────────────────────────────────────────────────────────────────
raise NotImplementedError(
    "futures/season_team_totals/05_predict_futures.ipynb is a scaffold. Sections are planned in the markdown cells above; "
    "implementation is gated on (1) review of 00_data_audit.ipynb and (2) a GO verdict "
    "in futures/artifacts/data_audit.json."
)

## Conclusion and next steps

**Status: scaffold — nothing implemented, nothing decided.** This notebook has produced no result and
written no artifact.

**Gate in force:** `futures/artifacts/data_audit.json` reads **`GO-TIER-B`** (2026-08-03) under
`PREREGISTRATION.md` §10 Amendment 1 — §7 gates **A and B** only, `tier_c_open: false`. The frozen
fold sets are the headline (10 test seasons) and the mandatory A1.4 strict-subset sensitivity
(4 test seasons, underpowered); both are read from the artifact, never recomputed here.

**On implementation this notebook must:** follow the repo's markdown → code → inline-test structure
with an explanation above and an interpretation below **every** code cell; report every headline
number twice (headline and A1.4 sensitivity); name the benchmark an *archived market consensus of
unattributed sportsbook origin*; and carry the §7 language fence — no sides, probabilities against a
posted line, confidence tiers, EV, or profitability, and none of the words *bet*, *edge*, *lock*,
*value*, *play*.

**Next step:** implement the sections planned above, in order, after the preceding notebook in the
pipeline has run and its artifact exists.